# DiffCAM objective: distinctiveness versus reference spread

**Weeks:** 28 · **Core question:** Which feature direction makes a target unusually different from references?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

**Paper connection:** DiffCAM Section 3. It looks for a direction $w$ that separates a target embedding $z$ from a reference mean $\mu$ while reducing within-reference projected scatter. Define $d=z-\mu$, $S_\mu=dd^T$, and $S_v=\sum_i(z_i-\mu)(z_i-\mu)^T$.

The Rayleigh quotient is $J(w)=(w^T S_\mu w)/(w^T S_v w)$. Unregularized $S_v$ may be singular. First explore the objective in two dimensions; only then solve it exactly.

**Learning goal:** Define all quantities in the objective and justify covariance-awareness.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
from diffcam_edu import make_tabular_cells
from diffcam_edu.math import reference_scatter,diffcam_weights
X,y=make_tabular_cells(80,seed=12);refs=X[y==0]
z=np.array([7.3,3.7]);mu,S=reference_scatter(refs);delta=z-mu
angles=np.linspace(0,np.pi,360)
regularization=2.
def J(w):return (w@delta)**2/(w@(S+regularization*np.eye(2))@w)
scores=np.array([J([np.cos(t),np.sin(t)]) for t in angles])
print('Best grid direction:',np.array([np.cos(angles[scores.argmax()]),np.sin(angles[scores.argmax()])]))

### Experiment 2 of 3

In [ ]:
w=diffcam_weights(z,refs,ridge=regularization,unit_norm=True)
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].scatter(*refs.T,alpha=.3);axs[0].scatter(*z,c='red',marker='*',s=140)
axs[0].arrow(*z,*(w*2),color='black',width=.02)
axs[0].set_title('Regularized best direction')
axs[1].plot(np.degrees(angles),scores);axs[1].set(xlabel='Angle',ylabel='Objective')
plt.show()
print('Analytic objective:',J(w),'Best grid objective:',scores.max())

### Experiment 3 of 3

In [ ]:
assert J(w)>=scores.max()-1e-4
print('Rayleigh quotient is unchanged by positive rescaling:',J(w),J(5*w))

## Exercises — complete independently
1. Remove ridge and investigate whether results change in 2D; then revisit a singular 20D example.
2. Explain why $w$ and $3w$ have equal quotient scores but could yield saliency maps with different ranges.

## Check your understanding

**Exit ticket:** Define all quantities in the objective and justify covariance-awareness.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)